# Hyperparameter Ablation Study : SimpleNetAugDR-3

In this notebook, we train the proposed **SimpleNetAugDR-3** architecture on **PlantVillage (tomato, 128×128)** the same setup as Table II and varies **one factor at a time** around the adopted
baseline `Adam, lr=1e-3, batch=16`:

| Factor | Values tested | Held fixed |
|---|---|---|
| Learning rate | `{1e-2, 1e-3, 1e-4}` | Adam, batch 16 |
| Batch size | `{16, 32, 64}` | Adam, lr 1e-3 |
| Optimizer | `{Adam, RMSprop, SGD(0.9)}` | lr 1e-3, batch 16 |

Seeds are reset before every run so all configs start from **identical weight
initialisation**; each reported difference is therefore due to the hyperparameter alone.

**Methodology (proper three-way split).** `train/` is split (stratified) into a **training** and a **validation** set; the ablation **selects** hyperparameters on the validation set (matching the paper's Table III). `valid/` is kept as a **held-out test set** that is *not* used during selection — the chosen configuration is confirmed on it once at the end.

## 1. Imports & setup

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   # silence TF info/warning C++ logs

import gc
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dropout, Dense
from tensorflow.keras.utils import to_categorical, normalize
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils import shuffle
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')          # silence Python-side retracing warnings

RANDOM_SEED = 123
IMG_SIZE    = (128, 128)     # same as the paper
NUM_CLASSES = 10

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print('TensorFlow', tf.__version__, '| ready.')

## 2. Proposed model : SimpleNetAugDR-3

In [ ]:
def create_model():
    model = Sequential([
        Conv2D(64, (4, 4), activation='relu', input_shape=(128, 128, 3)),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (4, 4), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (3, 3), activation='relu'),   # SimpleNetAugDR-3 = 64/64/64 (matches Figure 2 & benchmark NB01)
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='softmax'),
    ])
    model.compile(loss='categorical_crossentropy',
                  optimizer=Adam(learning_rate=0.001),
                  metrics=['accuracy'])
    return model


_m = create_model()
_m.summary()

print(f'\nParameters: {_m.count_params():,}  (SimpleNetAugDR-3, 64/64/64 = flatten 13x13x64, matches Figure 2)')
del _m; gc.collect()

## 3. Load PlantVillage (tomato, 128×128) — train / validation / test

`train/` is loaded and split (stratified) into a **training** and a **validation** set (`VAL_SPLIT`, default 20%). `valid/` is loaded and kept as a **held-out test set**. The ablation below trains on the training set, early-stops on the validation set, and reports each configuration's metrics on the **validation** set (used for hyperparameter selection). The test set is only touched once, at the very end, to confirm the selected configuration.

In [ ]:
# Adjust these paths to your environment if needed.
tomato_train = './new-plant-diseases-dataset/train/'
tomato_valid = './new-plant-diseases-dataset/valid/'

VAL_SPLIT = 0.2   # fraction of train/ held out as the validation split

from sklearn.model_selection import train_test_split


def load_pv(dir_path, img_size=IMG_SIZE):
    """Load PlantVillage tomato images (10 classes) — same pipeline as the paper."""
    X, Y, i = [], [], 0
    for path in tqdm(sorted(os.listdir(dir_path)),
                     desc=os.path.basename(dir_path.rstrip('/'))):
        if not path.startswith('.') and 'Tomato__' in path:
            sub = os.path.join(dir_path, path)
            for f in os.listdir(sub):
                fp = os.path.join(sub, f)
                if not f.startswith('.') and os.path.isfile(fp):
                    img = cv2.imread(fp)
                    img = Image.fromarray(img, 'RGB').resize(img_size)
                    X.append(np.array(img)); Y.append(i)
            i += 1
    return np.array(X, dtype=np.float32), np.array(Y)


# Proper 3-way split:
#   train/ -> TRAIN + VALIDATION (stratified)  | validation drives hyperparameter selection
#   valid/ -> held-out TEST set                | only used to confirm the selected config
X_pool, y_pool = load_pv(tomato_train)
X_te,   y_te   = load_pv(tomato_valid)

X_tr_r, X_va_r, y_tr_r, y_va_r = train_test_split(
    X_pool, y_pool, test_size=VAL_SPLIT, stratify=y_pool, random_state=RANDOM_SEED)

X_tr_r, y_tr_r = shuffle(X_tr_r, y_tr_r, random_state=RANDOM_SEED)
X_va_r, y_va_r = shuffle(X_va_r, y_va_r, random_state=RANDOM_SEED)
X_te,   y_te   = shuffle(X_te,   y_te,   random_state=RANDOM_SEED)

X_train = normalize(X_tr_r, axis=1).astype(np.float32); y_train = to_categorical(y_tr_r, NUM_CLASSES)
X_val   = normalize(X_va_r, axis=1).astype(np.float32); y_val   = to_categorical(y_va_r, NUM_CLASSES)
X_test  = normalize(X_te,   axis=1).astype(np.float32); y_test  = to_categorical(y_te,   NUM_CLASSES)
del X_pool, y_pool, X_tr_r, X_va_r, y_tr_r, y_va_r, X_te, y_te; gc.collect()

print(f'\nTrain: {X_train.shape} | Val: {X_val.shape} | Test (held-out): {X_test.shape}')


## 4. Ablation configuration & helpers

Each configuration is trained on the **training** set, early-stopped on the **validation** set, and scored on the **validation** set — validation is the selection signal for the one-factor-at-a-time sweeps (learning rate, batch size, optimizer). The held-out test set is not used here.

In [ ]:
# ── Baseline = the configuration adopted in the paper ────────────────────────
BASE_OPT   = 'Adam'
BASE_LR    = 1e-3
BASE_BATCH = 16

# ── Search spaces (one factor varied at a time) ──────────────────────────────
LR_GRID    = [1e-2, 1e-3, 1e-4]
BATCH_GRID = [16, 32, 64]
OPT_GRID   = ['Adam', 'RMSprop', 'SGD']

MAX_EPOCHS = 100
PATIENCE   = 5

# False -> full paper-grade run (~30–60 min). True -> ~2-min smoke test only.
QUICK_TEST = False


def make_optimizer(name, lr):
    if name == 'Adam':    return Adam(learning_rate=lr)
    if name == 'RMSprop': return RMSprop(learning_rate=lr)
    if name == 'SGD':     return SGD(learning_rate=lr, momentum=0.9)
    raise ValueError(f'Unknown optimizer: {name}')


def _predict_labels(model, X, batch=256):
    """Eager, batched inference — avoids predict() retracing and large-batch OOM."""
    out = []
    for i in range(0, len(X), batch):
        out.append(model(X[i:i + batch], training=False).numpy())
    return np.concatenate(out).argmax(axis=1)


def run_experiment(opt_name, lr, batch_size, X_tr, y_tr, X_va, y_va,
                   max_epochs=MAX_EPOCHS, patience=PATIENCE, seed=RANDOM_SEED):
    """Train SimpleNetAugDR-3 once; return VALIDATION metrics (the selection signal).

    Trains on (X_tr, y_tr), early-stops on (X_va, y_va), and scores on (X_va, y_va).
    The held-out test set is intentionally not seen here.
    """
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.backend.clear_session()

    model = create_model()                       # identical proposed architecture
    model.compile(loss='categorical_crossentropy',
                  optimizer=make_optimizer(opt_name, lr),
                  metrics=['accuracy'])

    es = EarlyStopping(monitor='val_loss', patience=patience,
                       restore_best_weights=True)

    t0 = time.time()
    hist = model.fit(X_tr, y_tr, batch_size=batch_size, epochs=max_epochs,
                     validation_data=(X_va, y_va), callbacks=[es], verbose=0)
    train_time = time.time() - t0

    y_pred = _predict_labels(model, X_va)
    y_true = y_va.argmax(axis=1)
    res = {
        'Optimizer':      opt_name,
        'Learning Rate':  lr,
        'Batch Size':     batch_size,
        'Accuracy (%)':   round(accuracy_score(y_true, y_pred) * 100, 2),
        'Precision (%)':  round(precision_score(y_true, y_pred, average='macro', zero_division=0) * 100, 2),
        'Recall (%)':     round(recall_score(y_true, y_pred, average='macro', zero_division=0) * 100, 2),
        'F1-Score (%)':   round(f1_score(y_true, y_pred, average='macro', zero_division=0) * 100, 2),
        'Epochs':         len(hist.history['loss']),
        'Train Time (s)': round(train_time, 1),
        'Params':         model.count_params(),
    }
    del model, hist; gc.collect(); tf.keras.backend.clear_session()
    return res


print('Helpers ready.')
print(f'Baseline: {BASE_OPT}, lr={BASE_LR:g}, batch={BASE_BATCH}  |  QUICK_TEST={QUICK_TEST}')

## 5. Run the ablation

In [ ]:
Xtr, ytr = X_train, y_train
Xva, yva = X_val,   y_val

if QUICK_TEST:
    from sklearn.model_selection import train_test_split as _tts
    idx, _ = _tts(np.arange(len(Xtr)), train_size=min(2000, len(Xtr) - 1),
                  stratify=ytr.argmax(1), random_state=RANDOM_SEED)
    Xtr, ytr = Xtr[idx], ytr[idx]
    MAX_EPOCHS_RUN, PATIENCE_RUN = 6, 3
    print(f'*** QUICK_TEST ON — {len(Xtr)} images, {MAX_EPOCHS_RUN} epochs. NOT paper results. ***')
else:
    MAX_EPOCHS_RUN, PATIENCE_RUN = MAX_EPOCHS, PATIENCE
    print('FULL RUN — paper-grade results.')


def _is_base(r):
    return (r['Optimizer'] == BASE_OPT and r['Learning Rate'] == BASE_LR
            and r['Batch Size'] == BASE_BATCH)


# Baseline is shared by all three sweeps -> run once, reuse.
print('\n[baseline] Adam | lr=1e-3 | batch=16')
base = run_experiment(BASE_OPT, BASE_LR, BASE_BATCH, Xtr, ytr, Xva, yva,
                      max_epochs=MAX_EPOCHS_RUN, patience=PATIENCE_RUN)
print(f'  -> acc {base["Accuracy (%)"]:.2f}%  ({base["Epochs"]} epochs)')

rows = []
for lr in LR_GRID:
    if lr == BASE_LR:
        r = dict(base)
    else:
        print(f'\n[lr] Adam | lr={lr:g} | batch=16')
        r = run_experiment(BASE_OPT, lr, BASE_BATCH, Xtr, ytr, Xva, yva,
                           max_epochs=MAX_EPOCHS_RUN, patience=PATIENCE_RUN)
        print(f'  -> acc {r["Accuracy (%)"]:.2f}%  ({r["Epochs"]} epochs)')
    r['Sweep'] = 'learning_rate'; rows.append(r)

for bs in BATCH_GRID:
    if bs == BASE_BATCH:
        r = dict(base)
    else:
        print(f'\n[batch] Adam | lr=1e-3 | batch={bs}')
        r = run_experiment(BASE_OPT, BASE_LR, bs, Xtr, ytr, Xva, yva,
                           max_epochs=MAX_EPOCHS_RUN, patience=PATIENCE_RUN)
        print(f'  -> acc {r["Accuracy (%)"]:.2f}%  ({r["Epochs"]} epochs)')
    r['Sweep'] = 'batch_size'; rows.append(r)

for opt in OPT_GRID:
    if opt == BASE_OPT:
        r = dict(base)
    else:
        print(f'\n[optimizer] {opt} | lr=1e-3 | batch=16')
        r = run_experiment(opt, BASE_LR, BASE_BATCH, Xtr, ytr, Xva, yva,
                           max_epochs=MAX_EPOCHS_RUN, patience=PATIENCE_RUN)
        print(f'  -> acc {r["Accuracy (%)"]:.2f}%  ({r["Epochs"]} epochs)')
    r['Sweep'] = 'optimizer'; rows.append(r)

ablation_df = pd.DataFrame(rows)
ablation_df['Baseline'] = ablation_df.apply(_is_base, axis=1)
ablation_df.to_csv('ablation_results_full.csv', index=False)
print('\nSaved -> ablation_results_full.csv')
ablation_df[['Sweep', 'Optimizer', 'Learning Rate', 'Batch Size', 'Accuracy (%)',
             'Precision (%)', 'Recall (%)', 'F1-Score (%)', 'Epochs', 'Train Time (s)', 'Baseline']]

## 6. Figure

In [ ]:
lr_df  = ablation_df[ablation_df.Sweep == 'learning_rate'].sort_values('Learning Rate', ascending=False)
bs_df  = ablation_df[ablation_df.Sweep == 'batch_size'].sort_values('Batch Size')
opt_df = ablation_df[ablation_df.Sweep == 'optimizer']

plt.rcParams.update({'font.size': 12, 'axes.titlesize': 13, 'axes.labelsize': 12})
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))

axes[0].bar([f'{v:g}' for v in lr_df['Learning Rate']], lr_df['Accuracy (%)'],
            color=['#4C72B0' if v == BASE_LR else '#A6BDDB' for v in lr_df['Learning Rate']])
axes[0].set_title('(a) Learning rate'); axes[0].set_xlabel('Learning rate')
axes[0].set_ylabel('Validation accuracy (%)')

axes[1].bar([str(int(v)) for v in bs_df['Batch Size']], bs_df['Accuracy (%)'],
            color=['#55A868' if v == BASE_BATCH else '#B7E1B0' for v in bs_df['Batch Size']])
axes[1].set_title('(b) Batch size'); axes[1].set_xlabel('Batch size')

axes[2].bar(opt_df['Optimizer'], opt_df['Accuracy (%)'],
            color=['#C44E52' if v == BASE_OPT else '#E7A9AB' for v in opt_df['Optimizer']])
axes[2].set_title('(c) Optimizer'); axes[2].set_xlabel('Optimizer')

for ax in axes:
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha='center', va='bottom', fontsize=10)
    ax.set_ylim(0, 100); ax.grid(axis='y', alpha=0.3)

fig.suptitle('Hyperparameter ablation on PlantVillage (SimpleNetAugDR-3) — adopted baseline highlighted', y=1.02)
plt.tight_layout()
plt.savefig('ablation_hyperparams.png', dpi=300, bbox_inches='tight')
plt.savefig('ablation_hyperparams.pdf', bbox_inches='tight')
plt.show()
print('Saved -> ablation_hyperparams.png / ablation_hyperparams.pdf')

## 7. Held-out test confirmation of the selected configuration

Hyperparameters were selected on the validation split above. Here the best configuration (highest validation accuracy) is retrained on the training set, early-stopped on the validation set, and evaluated **once** on the held-out test set — a leakage-free generalisation check. (The paper's Table III itself reports the validation numbers above.)

In [ ]:
# Pick the configuration with the highest VALIDATION accuracy from the ablation.
best_row   = ablation_df.loc[ablation_df['Accuracy (%)'].idxmax()]
best_opt   = best_row['Optimizer']
best_lr    = float(best_row['Learning Rate'])
best_batch = int(best_row['Batch Size'])
print(f'Selected by validation: {best_opt} | lr={best_lr:g} | batch={best_batch} '
      f'(val acc {best_row["Accuracy (%)"]:.2f}%)')

# Retrain on TRAIN, early-stop on VALIDATION, evaluate once on the held-out TEST set.
np.random.seed(RANDOM_SEED); tf.random.set_seed(RANDOM_SEED); tf.keras.backend.clear_session()
final_model = create_model()
final_model.compile(loss='categorical_crossentropy',
                    optimizer=make_optimizer(best_opt, best_lr), metrics=['accuracy'])
final_model.fit(X_train, y_train, batch_size=best_batch, epochs=MAX_EPOCHS_RUN,
                validation_data=(X_val, y_val),
                callbacks=[EarlyStopping(monitor='val_loss', patience=PATIENCE_RUN,
                                         restore_best_weights=True)],
                verbose=0)

y_pred_test = _predict_labels(final_model, X_test)
y_true_test = y_test.argmax(axis=1)
test_metrics = {
    'Accuracy (%)':  round(accuracy_score(y_true_test, y_pred_test) * 100, 2),
    'Precision (%)': round(precision_score(y_true_test, y_pred_test, average='macro', zero_division=0) * 100, 2),
    'Recall (%)':    round(recall_score(y_true_test, y_pred_test, average='macro', zero_division=0) * 100, 2),
    'F1-Score (%)':  round(f1_score(y_true_test, y_pred_test, average='macro', zero_division=0) * 100, 2),
}
print('\nHeld-out TEST metrics for the selected configuration:')
for k, v in test_metrics.items():
    print(f'  {k:14s}{v}')
pd.DataFrame([{'Optimizer': best_opt, 'Learning Rate': best_lr, 'Batch Size': best_batch,
               **test_metrics}]).to_csv('selected_config_test_metrics.csv', index=False)
del final_model; gc.collect(); tf.keras.backend.clear_session()
